In [1]:
__author__ = "Melat Ghebreselassie"
__version__ = "01/22/2026"

# Table of Contents  
1. [Set-up](#Set-up)     
2. [pyvene 101 with NDIF Backend](#pyvene-101-with-ndif-backend)
3. [Collecting Interventions](#Collecting-Interventions)
4. [Vanilla Intervention](#Vanilla-Intervention)
5. [Trainable Interventions](#Trainable-Interventions-via-pyvene-API)
   - LowRankRotatedSpaceIntervention
   - RotatedSpaceIntervention
   - BoundlessRotatedSpaceIntervention
   - SigmoidMaskRotatedSpaceIntervention
   - SigmoidMaskIntervention
6. [Summary](#Summary)

# Set-up

In [ ]:
import os

# Load credentials from a .env file (recommended - keep .env out of git)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded credentials from .env")
except ImportError:
    print("python-dotenv not installed; falling back to environment variables.")

# Verify required keys are present
assert os.environ.get("NDIF_API_KEY"), "Set NDIF_API_KEY in your .env file or environment"
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in your .env file or environment"
print("Credentials OK")

In [ ]:
try:
    # This library is our indicator that the required installs
    # need to be done.
    import pyvene as pv
    import nnsight 

except ModuleNotFoundError:
    !pip install git+https://github.com/stanfordnlp/pyvene.git

# pyvene 101 with ndif backend


In [3]:
USE_REMOTE = True

In [4]:
from nnsight import LanguageModel

print("Loading GPT-2 model...")
if USE_REMOTE:
    model = LanguageModel('openai-community/gpt2')
else:
    model = LanguageModel('openai-community/gpt2', device_map='cpu')

tokenizer = model.tokenizer

Loading GPT-2 model...


# Collecting Interventions

In [ ]:
import pyvene as pv

base_text = "When John and Mary went to the shops, Mary gave the bag to"

pv_model = pv.build_intervenable_model({
        "component": "transformer.h[0].mlp.c_proj.output",
        "intervention": pv.CollectIntervention()
    }, model=model, 
    remote=USE_REMOTE)

collected_mlp_out = pv_model(
    base = tokenizer(base_text, return_tensors="pt"),
    unit_locations={"base": [h for h in range(12)]}
)[0][-1][0]

# Resolve the saved proxy value
if hasattr(collected_mlp_out, 'value'):
    collected_mlp_out = collected_mlp_out.value

print(f"Collected MLP output shape: {collected_mlp_out.shape}")
print("CollectIntervention: SUCCESS!" if collected_mlp_out is not None else "FAILED")

# Vanilla Intervention

In [6]:
def get_clean_output(text):
    """Get clean model output without any intervention."""
    with model.session(remote=USE_REMOTE):
        with model.trace(text):
            output = model.lm_head.output.save()  # Use lm_head.output for logits
    return output


def get_logits(output):
    """Extract logits from various output formats."""
    if hasattr(output, 'logits'):
        return output.logits
    elif isinstance(output, dict) and 'logits' in output:
        return output['logits']
    elif hasattr(output, 'value'):
        return output.value.logits if hasattr(output.value, 'logits') else output.value
    return output

In [ ]:

# Prepare inputs FIRST
# Note: GPT-2's baseline prediction for both Spain/Italy is often ' the' (not the city name).
# The intervention swaps layer-0 activations from Italy into the Spain forward pass,
# which should shift the prediction toward ' Rome'.
base_text = "The capital of Spain is"
source_text = "The capital of Italy is"

pv_model = pv.build_intervenable_model({
        "component": "transformer.h[0].output",
        "intervention": pv.VanillaIntervention()
}, model=model, remote=USE_REMOTE)

# Now get clean output with the correct base_text
clean_output = get_clean_output(base_text)
clean_logits = get_logits(clean_output)

# Use raw strings for remote compatibility
_, intervened = pv_model(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
intervened_logits = get_logits(intervened)

clean_pred = tokenizer.decode(clean_logits[0, -1].argmax())
intervened_pred = tokenizer.decode(intervened_logits[0, -1].argmax())
logit_diff = (clean_logits - intervened_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"Intervened prediction: '{intervened_pred}'")
print(f"Mean absolute logit difference: {logit_diff:.4f}")
# Success = intervention changed the output toward Rome (Italy), regardless of clean baseline
print("SUCCESS!" if intervened_pred == " Rome" else "UNEXPECTED OUTPUT")

# Trainable Interventions via pyvene API

Trainable interventions like `LowRankRotatedSpaceIntervention`, `RotatedSpaceIntervention`, and `BoundlessRotatedSpaceIntervention` can be used directly via the pyvene API - no manual nnsight operations required!

In [8]:
import torch

EMBED_DIM = 768  # GPT-2 hidden dimension

# Use same text as VanillaIntervention example
base_text = "The capital of Spain is"
source_text = "The capital of Italy is"

# Get clean output for comparison
clean_output = get_clean_output(base_text)
clean_logits = get_logits(clean_output)

# =============================================================================
# LowRankRotatedSpaceIntervention via pyvene API (simple approach)
# =============================================================================
intervention = pv.LowRankRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_lowrank = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": intervention
}, model=model, remote=USE_REMOTE)

_, lowrank_output = pv_lowrank(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
lowrank_logits = get_logits(lowrank_output)

clean_pred = tokenizer.decode(clean_logits[0, -1].argmax())
lowrank_pred = tokenizer.decode(lowrank_logits[0, -1].argmax())
logit_diff = (clean_logits - lowrank_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"LowRankRotatedSpace prediction: '{lowrank_pred}'")
print(f"Mean logit diff: {logit_diff:.4f}")
print(f"Intervention had effect: {logit_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/504k [00:00<?]

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
LowRankRotatedSpace prediction: ' Madrid'
Mean logit diff: 0.2520
Intervention had effect: True


In [9]:
# =============================================================================
# RotatedSpaceIntervention via pyvene API
# =============================================================================
rotated_intervention = pv.RotatedSpaceIntervention(embed_dim=EMBED_DIM)

pv_rotated = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": rotated_intervention
}, model=model, remote=USE_REMOTE)

_, rotated_output = pv_rotated(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
rotated_logits = get_logits(rotated_output)

rotated_pred = tokenizer.decode(rotated_logits[0, -1].argmax())
rotated_diff = (clean_logits - rotated_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"RotatedSpace prediction: '{rotated_pred}'")
print(f"Mean logit diff: {rotated_diff:.4f}")
print(f"Intervention had effect: {rotated_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
RotatedSpace prediction: ' Rome'
Mean logit diff: 1.5078
Intervention had effect: True


In [ ]:
# =============================================================================
# BoundlessRotatedSpaceIntervention via pyvene API
# =============================================================================
boundless_intervention = pv.BoundlessRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_boundless = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": boundless_intervention
}, model=model, remote=USE_REMOTE)

_, boundless_output = pv_boundless(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
boundless_logits = get_logits(boundless_output)

boundless_pred = tokenizer.decode(boundless_logits[0, -1].argmax())
boundless_diff = (clean_logits - boundless_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"BoundlessRotatedSpace prediction: '{boundless_pred}'")
print(f"Mean logit diff: {boundless_diff:.4f}")
# At random init the boundary (sigmoid) softly blends base and source, so
# logit diff > 0 confirms the intervention is running even if top-1 doesn't change.
print(f"Intervention had effect: {boundless_diff > 0.01} (top-1 may match clean at random init)")

In [11]:
# =============================================================================
# SigmoidMaskRotatedSpaceIntervention via pyvene API
# =============================================================================
sigmoid_mask_rotated_intervention = pv.SigmoidMaskRotatedSpaceIntervention(
    embed_dim=EMBED_DIM, low_rank_dimension=64
)

pv_sigmoid_mask_rotated = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": sigmoid_mask_rotated_intervention
}, model=model, remote=USE_REMOTE)

_, sigmoid_mask_rotated_output = pv_sigmoid_mask_rotated(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
sigmoid_mask_rotated_logits = get_logits(sigmoid_mask_rotated_output)

sigmoid_mask_rotated_pred = tokenizer.decode(sigmoid_mask_rotated_logits[0, -1].argmax())
sigmoid_mask_rotated_diff = (clean_logits - sigmoid_mask_rotated_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"SigmoidMaskRotatedSpace prediction: '{sigmoid_mask_rotated_pred}'")
print(f"Mean logit diff: {sigmoid_mask_rotated_diff:.4f}")
print(f"Intervention had effect: {sigmoid_mask_rotated_diff > 0.01}")

⬇ Downloading:   0%|          | 0.00/512k [00:00<?]

Clean prediction: ' the'
SigmoidMaskRotatedSpace prediction: ' Rome'
Mean logit diff: 1.1250
Intervention had effect: True


In [ ]:
# =============================================================================
# SigmoidMaskIntervention via pyvene API
# (Learnable mask without rotation - applies directly in activation space)
# =============================================================================
sigmoid_mask_intervention = pv.SigmoidMaskIntervention(embed_dim=EMBED_DIM)

pv_sigmoid_mask = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": sigmoid_mask_intervention
}, model=model, remote=USE_REMOTE)

_, sigmoid_mask_output = pv_sigmoid_mask(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
sigmoid_mask_logits = get_logits(sigmoid_mask_output)

sigmoid_mask_pred = tokenizer.decode(sigmoid_mask_logits[0, -1].argmax())
sigmoid_mask_diff = (clean_logits - sigmoid_mask_logits).abs().mean().item()

print(f"Clean prediction: '{clean_pred}'")
print(f"SigmoidMask prediction: '{sigmoid_mask_pred}'")
print(f"Mean logit diff: {sigmoid_mask_diff:.4f}")
# At random init mask=zeros → sigmoid(0/temp)=0.5, so output is 50/50 blend of base and source.
# logit diff > 0 confirms the intervention is running even if top-1 doesn't change.
print(f"Intervention had effect: {sigmoid_mask_diff > 0.01} (top-1 may match clean at random init)")

# Summary

All intervention types work via the standard pyvene API with NDIF backend (`remote=True`):

## Non-Trainable Interventions

| Intervention | Description | NDIF Support |
|-------------|-------------|--------------|
| `CollectIntervention` | Collects activations without modification | ✓ Works |
| `VanillaIntervention` | Simple activation swapping (base ← source) | ✓ Works |

## Trainable Interventions

| Intervention | Description | NDIF Support |
|-------------|-------------|--------------|
| `LowRankRotatedSpaceIntervention` | Projects to low-rank rotated subspace, swaps, projects back | ✓ Works |
| `RotatedSpaceIntervention` | Full-rank rotation with learnable orthogonal matrix | ✓ Works |
| `BoundlessRotatedSpaceIntervention` | Rotation + learnable boundary for soft intervention | ✓ Works |
| `SigmoidMaskRotatedSpaceIntervention` | Rotation + per-dimension sigmoid masks | ✓ Works |
| `SigmoidMaskIntervention` | Per-dimension sigmoid masks (no rotation) | ✓ Works |

## Usage Pattern

```python
# All interventions follow the same pattern:
intervention = pv.LowRankRotatedSpaceIntervention(embed_dim=768, low_rank_dimension=64)

pv_model = pv.build_intervenable_model({
    "component": "transformer.h[0].output",
    "intervention": intervention
}, model=model, remote=True)

_, output = pv_model(
    base=base_text,
    sources=[source_text],
    unit_locations={"sources->base": ([None], [None])}
)
```

No manual nnsight operations needed - just use `pv.build_intervenable_model()` with `remote=True`!